# Influencerinnen-Auswahlabbildung

Erzeugt die Abbildung und Metadaten zur Auswahl der untersuchten Influencerinnen.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
__file__ = str(PROJECT_ROOT / "exports" / "notebooks" / "04_influencerinnen_auswahl_notebook.py")
print(f"Projektordner: {PROJECT_ROOT}")


## Code


In [ ]:
from __future__ import annotations

import csv
import re
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps


ROOT = Path(__file__).resolve().parents[2]
DATA_CSV = ROOT / "data" / "04_analysis_results" / "visual_features" / "99_AI_AND_REAL_TIKTOK_VIDEOS_all_results_merged.csv"
FRAME_DIR = ROOT / "exports" / "examples" / "frames"
OUT_DIR = ROOT / "exports" / "examples" / "figures"
OUT_PNG = OUT_DIR / "abb_influencerinnen_auswahl.png"
OUT_META = OUT_DIR / "abb_influencerinnen_auswahl_metadata.csv"

TILE_W = 300
TILE_H = 533
LABEL_H = 112
GAP = 22
MARGIN = 36
COLS = 5

BG = (248, 247, 242)
PANEL_BG = (255, 255, 255)
TEXT = (28, 31, 35)
MUTED = (78, 84, 93)
VI = (105, 73, 160)
RI = (32, 118, 117)


def font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        Path("C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf"),
        Path("C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf"),
        Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf"),
        Path("/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size)
    return ImageFont.load_default()


FONT_TITLE = font(18, bold=True)
FONT_SMALL = font(14)
FONT_TINY = font(12)

ACCOUNT_TIERS = {
    "lilmiquela": "Mega Virtual",
    "imma.tokyo": "Macro Virtual",
    "millas_sofia": "Micro Virtual",
    "brexleyai": "Micro Virtual",
    "ai.kalai": "Nano Virtual",
    "jessicawangofficial": "Mega Real",
    "ericanic0le": "Macro Real",
    "hollybrandmusic": "Micro Real",
    "nobodywhoareu": "Nano/Micro Real",
    "misshannahashton": "Nano Real",
}

TIER_ORDER = {
    "Nano": 0,
    "Nano/Micro": 0.5,
    "Micro": 1,
    "Mid-Tier": 2,
    "Macro": 3,
    "Mega": 4,
}


def video_id_from_frame(path: Path) -> str | None:
    match = re.search(r"_(\d{18,20})\.jpe?g$", path.name)
    return match.group(1) if match else None


def format_metric(value) -> str:
    if pd.isna(value):
        return "NA"
    number = float(value)
    if number >= 1_000_000:
        text = f"{number / 1_000_000:.1f}".replace(".", ",")
        return f"{text} Mio."
    if number >= 1_000:
        return f"{int(round(number)):,}".replace(",", ".")
    return str(int(round(number)))


def draw_wrapped(draw: ImageDraw.ImageDraw, xy: tuple[int, int], text: str, max_width: int, fnt, fill=TEXT, max_lines: int = 2) -> int:
    words = str(text).split()
    lines: list[str] = []
    line = ""
    for word in words:
        candidate = f"{line} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=fnt)[2] <= max_width or not line:
            line = candidate
        else:
            lines.append(line)
            line = word
    if line:
        lines.append(line)

    x, y = xy
    line_height = draw.textbbox((0, 0), "Ag", font=fnt)[3] + 6
    for visible_line in lines[:max_lines]:
        draw.text((x, y), visible_line, font=fnt, fill=fill)
        y += line_height
    return y


def frame_lookup() -> dict[str, Path]:
    frames: dict[str, Path] = {}
    for path in sorted(FRAME_DIR.glob("*.jpeg")):
        video_id = video_id_from_frame(path)
        if video_id and video_id not in frames:
            frames[video_id] = path
    return frames


def pick_examples() -> pd.DataFrame:
    frames = frame_lookup()
    df = pd.read_csv(DATA_CSV, dtype={"video_id": str})
    df = df[df["video_id"].isin(frames)].copy()
    df["_frame_path"] = df["video_id"].map(frames)
    df["_sort_type"] = df["influencer_type"].map({"ai": 0, "real": 1}).fillna(2)
    df["_tier"] = df["author_username"].map(ACCOUNT_TIERS)
    df["_tier_order"] = df["_tier"].fillna("").map(
        lambda value: TIER_ORDER.get(str(value).rsplit(" ", 1)[0], 99)
    )
    df = df.sort_values(
        ["_sort_type", "_tier_order", "video_view_count", "video_like_count", "author_username"],
        ascending=[True, True, False, False, True],
    )
    examples = df.groupby("author_username", as_index=False).first()
    return examples.sort_values(["_sort_type", "_tier_order", "video_view_count"], ascending=[True, True, False])


def render(examples: pd.DataFrame) -> None:
    rows = (len(examples) + COLS - 1) // COLS
    width = MARGIN * 2 + COLS * TILE_W + (COLS - 1) * GAP
    height = MARGIN * 2 + rows * (TILE_H + LABEL_H) + (rows - 1) * GAP
    canvas = Image.new("RGB", (width, height), BG)
    draw = ImageDraw.Draw(canvas)

    for idx, (_, row) in enumerate(examples.iterrows()):
        grid_row = idx // COLS
        grid_col = idx % COLS
        x = MARGIN + grid_col * (TILE_W + GAP)
        y = MARGIN + grid_row * (TILE_H + LABEL_H + GAP)

        image = Image.open(row["_frame_path"]).convert("RGB")
        image = ImageOps.fit(image, (TILE_W, TILE_H), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
        canvas.paste(image, (x, y))

        typ = "VI" if row["influencer_type"] == "ai" else "RI"
        stripe = VI if typ == "VI" else RI
        draw.rectangle((x, y + TILE_H, x + TILE_W, y + TILE_H + LABEL_H), fill=PANEL_BG)
        draw.rectangle((x, y + TILE_H, x + 8, y + TILE_H + LABEL_H), fill=stripe)

        title = f"@{row['author_username']}"
        display = str(row["author_displayname"]) if not pd.isna(row["author_displayname"]) else ""
        tier = ACCOUNT_TIERS.get(str(row["author_username"]), typ)
        metrics = (
            f"Views: {format_metric(row['video_view_count'])} | "
            f"Likes: {format_metric(row['video_like_count'])} | "
            f"Kommentare: {format_metric(row['video_comment_count'])}"
        )

        label_x = x + 18
        label_y = y + TILE_H + 8
        draw_wrapped(draw, (label_x, label_y), title, TILE_W - 28, FONT_TITLE, fill=TEXT, max_lines=1)
        draw.text((label_x, label_y + 26), f"{display} | {tier}", font=FONT_TINY, fill=MUTED)
        draw_wrapped(draw, (label_x, label_y + 48), metrics, TILE_W - 28, FONT_SMALL, fill=MUTED, max_lines=3)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    canvas.save(OUT_PNG, quality=95)


def write_metadata(examples: pd.DataFrame) -> None:
    fieldnames = [
        "position",
        "video_id",
        "type",
        "author_username",
        "author_displayname",
        "tier",
        "video_view_count",
        "video_like_count",
        "video_comment_count",
        "frame_path",
    ]
    with OUT_META.open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for pos, (_, row) in enumerate(examples.iterrows(), start=1):
            writer.writerow(
                {
                    "position": pos,
                    "video_id": row["video_id"],
                    "type": "VI" if row["influencer_type"] == "ai" else "RI",
                    "author_username": row["author_username"],
                    "author_displayname": row["author_displayname"],
                    "tier": ACCOUNT_TIERS.get(str(row["author_username"]), ""),
                    "video_view_count": row["video_view_count"],
                    "video_like_count": row["video_like_count"],
                    "video_comment_count": row["video_comment_count"],
                    "frame_path": Path(row["_frame_path"]).relative_to(ROOT),
                }
            )


def main() -> None:
    examples = pick_examples()
    if examples.empty:
        raise RuntimeError(f"No matching frames found in {FRAME_DIR}")
    render(examples)
    write_metadata(examples)
    print(f"Created {OUT_PNG}")
    print(f"Wrote {OUT_META}")



## Ausführung


In [ ]:
main()
